## Import Required API's

In [2]:

from pathlib import Path
import json
import re
import sqlite3
import statistics
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup

##### 1. Configuration

In [1]:

ROOT = Path.cwd()

OUTPUT = ROOT / "output"
SQL_OUTPUT = OUTPUT / "sql_outputs"

OUTPUT.mkdir(parents=True, exist_ok=True)
SQL_OUTPUT.mkdir(parents=True, exist_ok=True)

DB_PATH = OUTPUT / "books.db"
CSV_PATH = OUTPUT / "books_cleaned.csv"

print("Root directory:", ROOT)
print("Database path:", DB_PATH)
BASE_URL = "https://books.toscrape.com/"
CATALOGUE_URL = BASE_URL + "catalogue/page-{}.html"

GBP_TO_INR = 105.50
MAX_PAGES = 5

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "EducationalDataPipeline/1.0"
})

RATING_MAP = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5,
}

# 2. HTTP and parsing helpers

def get_soup(url):
    """Fetch a page and parse it, with basic error handling."""
    response = SESSION.get(url, timeout=30)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")

def parse_price(text):
    """Convert a GBP price string to float."""
    if not text:
        return None

    cleaned = text.replace("£", "").replace(",", "").strip()

    try:
        return float(cleaned)
    except (ValueError, TypeError):
        return None


def parse_rating(card):
    """Extract the star rating from the CSS class."""
    element = card.select_one(".star-rating")

    if not element:
        return None

    for word, number in RATING_MAP.items():
        if word in element.get("class", []):
            return number

    return None


def parse_stock(text):
    """Convert availability text into a boolean."""
    if not text:
        return None

    if re.search(r"\bin\s+stock\b", text, re.IGNORECASE):
        return True

    if re.search(r"\bout\s+of\s+stock\b", text, re.IGNORECASE):
        return False

    return None


def get_category(product_url):
    """Read the category from the product page breadcrumb."""
    soup = get_soup(product_url)

    breadcrumb = soup.select("ul.breadcrumb li a")

    # Breadcrumb normally contains:
    # Home > Books > Category > Product
    if len(breadcrumb) >= 3:
        return breadcrumb[-1].get_text(strip=True)

    return None

# 3. Scrape catalogue pages

def scrape_books():
    records = []
    seen_urls = set()

    for page_number in range(1, MAX_PAGES + 1):
        url = CATALOGUE_URL.format(page_number)
        print(f"Scraping catalogue page {page_number}: {url}")

        soup = get_soup(url)
        cards = soup.select("article.product_pod")

        if not cards:
            print("No product cards found; stopping.")
            break

        for card in cards:
            try:
                title_element = card.select_one("h3 a")
                price_element = card.select_one(".price_color")
                stock_element = card.select_one(".availability")

                if not title_element:
                    continue

                title = title_element.get("title", "").strip()
                relative_url = title_element.get("href", "")
                product_url = requests.compat.urljoin(url, relative_url)

                if product_url in seen_urls:
                    continue
                seen_urls.add(product_url)

                price_text = (
                    price_element.get_text(strip=True)
                    if price_element else ""
                )

                stock_text = (
                    stock_element.get_text(" ", strip=True)
                    if stock_element else ""
                )

                # Individual product page provides category.
                category = get_category(product_url)

                records.append({
                    "title": title,
                    "price": price_text,
                    "star_rating": (
                        next(
                            (word for word, number in RATING_MAP.items()
                             if number == parse_rating(card)),
                            None
                        )
                    ),
                    "availability": stock_text,
                    "category": category,
                })

                # Be polite to the practice website.
                time.sleep(0.1)

            except requests.RequestException as exc:
                print(f"Skipping product due to request error: {exc}")
                continue
            except Exception as exc:
                print(f"Skipping malformed product: {exc}")
                continue

    return pd.DataFrame(records)



# 4. Clean, validate, and convert


def clean_books(df):
    df = df.copy()

    # Preserve raw source fields while creating typed fields.
    df["price_gbp"] = df["price"].apply(parse_price)
    df["rating"] = df["star_rating"].map(RATING_MAP)
    df["in_stock"] = df["availability"].apply(parse_stock)

    # Normalize text fields.
    for column in ["title", "category"]:
        df[column] = df[column].astype("string").str.strip()

    df["title"] = df["title"].replace("", pd.NA)
    df["category"] = df["category"].replace("", pd.NA)

    # Rows without essential identifiers cannot be stored meaningfully.
    df = df.dropna(subset=["title", "category"])

    # Numeric parse failures: median imputation.
    for column in ["price_gbp", "rating"]:
        df[column] = pd.to_numeric(df[column], errors="coerce")
        median_value = df[column].median()

        if pd.isna(median_value):
            raise ValueError(
                f"Cannot impute {column}: no valid numeric values."
            )

        if column == "rating":
            median_value = round(median_value)

        df[column] = df[column].fillna(median_value)

    df["rating"] = df["rating"].round().astype("int64")

    # Rating must be in the range 1–5.
    df = df[df["rating"].between(1, 5)]

    # Stock parsing failures: drop those rows because stock status
    # is categorical and median imputation is not meaningful.
    df = df.dropna(subset=["in_stock"])
    df["in_stock"] = df["in_stock"].astype(bool)

    # Fixed project-defined conversion rate.
    df["price_inr"] = (
        df["price_gbp"] * GBP_TO_INR
    ).round(2)

    # Keep the required typed fields and source text for auditability.
    return df[
        [
            "title",
            "price",
            "price_gbp",
            "price_inr",
            "star_rating",
            "rating",
            "availability",
            "in_stock",
            "category",
        ]
    ].reset_index(drop=True)



# 5. Create normalized SQLite database


def create_database(df):
    if DB_PATH.exists():
        DB_PATH.unlink()

    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("PRAGMA foreign_keys = ON")

        conn.executescript("""
            CREATE TABLE categories (
                category_id INTEGER PRIMARY KEY,
                category_name TEXT NOT NULL UNIQUE
            );

            CREATE TABLE books (
                book_id INTEGER PRIMARY KEY,
                title TEXT NOT NULL,
                price_gbp REAL NOT NULL,
                price_inr REAL NOT NULL,
                rating INTEGER NOT NULL
                    CHECK (rating BETWEEN 1 AND 5),
                in_stock INTEGER NOT NULL
                    CHECK (in_stock IN (0, 1)),
                category_id INTEGER NOT NULL,
                FOREIGN KEY (category_id)
                    REFERENCES categories(category_id)
            );
        """)

        categories = sorted(df["category"].unique())

        conn.executemany(
            "INSERT INTO categories (category_name) VALUES (?)",
            [(category,) for category in categories]
        )

        category_lookup = pd.read_sql_query(
            "SELECT category_id, category_name FROM categories",
            conn
        ).set_index("category_name")["category_id"].to_dict()

        books_to_insert = []

        for row in df.itertuples(index=False):
            books_to_insert.append((
                row.title,
                float(row.price_gbp),
                float(row.price_inr),
                int(row.rating),
                int(row.in_stock),
                int(category_lookup[row.category]),
            ))

        conn.executemany("""
            INSERT INTO books (
                title, price_gbp, price_inr,
                rating, in_stock, category_id
            )
            VALUES (?, ?, ?, ?, ?, ?)
        """, books_to_insert)

        conn.commit()



# 6. SQL queries and saved outputs


QUERIES = {
    "01_select_where": """
        SELECT title, price_gbp, rating
        FROM books
        WHERE price_gbp > 20
        ORDER BY price_gbp DESC;
    """,

    "02_order_by_limit": """
        SELECT title, price_gbp, rating
        FROM books
        ORDER BY price_gbp DESC
        LIMIT 10;
    """,

    "03_distinct_categories": """
        SELECT DISTINCT c.category_name
        FROM books b
        JOIN categories c
          ON b.category_id = c.category_id
        ORDER BY c.category_name;
    """,

    "04_between": """
        SELECT title, price_gbp, price_inr
        FROM books
        WHERE price_gbp BETWEEN 10 AND 20
        ORDER BY price_gbp;
    """,

    "05_in_stock": """
        SELECT title, rating, in_stock
        FROM books
        WHERE rating IN (4, 5)
        ORDER BY rating DESC, title;
    """,

    "06_join_books_categories": """
        SELECT
            b.book_id,
            b.title,
            c.category_name,
            b.price_gbp,
            b.price_inr,
            b.rating,
            b.in_stock
        FROM books b
        JOIN categories c
          ON b.category_id = c.category_id
        ORDER BY c.category_name, b.title;
    """,
}


def execute_queries(df):
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("PRAGMA foreign_keys = ON")

        for name, query in QUERIES.items():
            result = pd.read_sql_query(query, conn)

            print(f"\n--- {name} ---")
            print(query.strip())
            print(result.to_string(index=False))

            result.to_csv(
                SQL_OUTPUT / f"{name}.csv",
                index=False
            )

            with open(
                SQL_OUTPUT / f"{name}.sql",
                "w",
                encoding="utf-8"
            ) as file:
                file.write(query.strip() + "\n")

        # Read back two query results with pd.read_sql.
        query1 = pd.read_sql_query(
            QUERIES["01_select_where"], conn
        )

        query2 = pd.read_sql_query(
            QUERIES["02_order_by_limit"], conn
        )

        # Read the JOIN result from SQL.
        sql_join = pd.read_sql_query(
            QUERIES["06_join_books_categories"], conn
        )

        # Reproduce the same JOIN using pandas.merge.
        categories_df = pd.read_sql_query(
            "SELECT category_id, category_name FROM categories",
            conn
        )

        books_df = pd.read_sql_query(
            "SELECT * FROM books",
            conn
        )

        pandas_join = pd.merge(
            books_df,
            categories_df,
            on="category_id",
            how="inner"
        )

        pandas_join = pandas_join[
            [
                "book_id",
                "title",
                "category_name",
                "price_gbp",
                "price_inr",
                "rating",
                "in_stock",
            ]
        ].sort_values(
            ["category_name", "title"]
        ).reset_index(drop=True)

        # SQL and pandas must have equivalent results.
        sql_join = sql_join.reset_index(drop=True)

        # SQLite stores booleans as 0/1; compare normalized values.
        sql_join["in_stock"] = sql_join["in_stock"].astype(int)
        pandas_join["in_stock"] = pandas_join["in_stock"].astype(int)

        assert sql_join.equals(pandas_join), (
            "SQL JOIN and pandas.merge results do not match."
        )

        print("\nSQL JOIN and pandas.merge outputs match.")
        print("\nSQL JOIN output:")
        print(sql_join.to_string(index=False))
        print("\nPandas merge output:")
        print(pandas_join.to_string(index=False))

        sql_join.to_csv(
            SQL_OUTPUT / "join_sql_result.csv", index=False
        )
        pandas_join.to_csv(
            SQL_OUTPUT / "join_pandas_result.csv", index=False
        )

        # Save a compact execution summary.
        summary = {
            "query_01_rows": len(query1),
            "query_02_rows": len(query2),
            "join_rows": len(sql_join),
            "sql_pandas_join_match": True,
        }

        with open(
            SQL_OUTPUT / "summary.json",
            "w",
            encoding="utf-8"
        ) as file:
            json.dump(summary, file, indent=2)



# 7. Main entry point


def main():
    print("Starting Books to Scrape pipeline...")

    raw_df = scrape_books()

    if raw_df.empty:
        raise RuntimeError("No books were scraped.")

    print(f"\nRaw scraped rows: {len(raw_df)}")
    print("Categories found:", raw_df["category"].nunique())

    cleaned_df = clean_books(raw_df)

    if len(cleaned_df) < 60:
        raise RuntimeError(
            f"Only {len(cleaned_df)} valid books remain. "
            "The assignment requires at least 60."
        )

    if cleaned_df["category"].nunique() < 3:
        raise RuntimeError(
            "Fewer than 3 categories found."
        )

    cleaned_df.to_csv(CSV_PATH, index=False)

    create_database(cleaned_df)
    execute_queries(cleaned_df)

    print("\nPipeline completed successfully.")
    print(f"Cleaned books: {len(cleaned_df)}")
    print(f"Categories: {cleaned_df['category'].nunique()}")
    print(f"Database: {DB_PATH}")
    print(f"Clean CSV: {CSV_PATH}")
    print(f"SQL outputs: {SQL_OUTPUT}")


if __name__ == "__main__":
    main()

Root directory: C:\Users\sreen\Masai_AIML_Telugu
Database path: C:\Users\sreen\Masai_AIML_Telugu\output\books.db
Starting Books to Scrape pipeline...
Scraping catalogue page 1: https://books.toscrape.com/catalogue/page-1.html
Scraping catalogue page 2: https://books.toscrape.com/catalogue/page-2.html
Scraping catalogue page 3: https://books.toscrape.com/catalogue/page-3.html
Scraping catalogue page 4: https://books.toscrape.com/catalogue/page-4.html
Scraping catalogue page 5: https://books.toscrape.com/catalogue/page-5.html

Raw scraped rows: 100
Categories found: 29


ValueError: Cannot impute price_gbp: no valid numeric values.